setup and the merge function

merge_into_gold(df, name, keys) works exactly like the silver one: it matches on the key columns, updates rows that exist, inserts rows that don’t, and prints how many of each.

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"

def silver(name):
    return spark.table(f"{catalog}.{schema}.silver_{name}")

def gold(name):
    return spark.table(f"{catalog}.{schema}.gold_{name}")

def merge_into_gold(df, name, keys):
    df.createOrReplaceTempView(f"src_{name}")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    res = spark.sql(f"""
        MERGE INTO {catalog}.{schema}.gold_{name} AS t
        USING src_{name} AS s
        ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    print(f"{name}: inserted {res['num_inserted_rows']}, updated {res['num_updated_rows']}, table total {gold(name).count()}")

reset the fact tables (run once)

This empties the two gold fact tables (the tables and their columns stay).

In [0]:
for t in ["fact_order_items", "fact_orders"]:
    spark.sql(f"TRUNCATE TABLE {catalog}.{schema}.gold_{t}")
    print(t, gold(t).count())

fact_order_items 0
fact_orders 0


attach the right customer version to each order

This is the point-in-time join. For each silver order, it finds the dim_customer row with the same customer_id whose validity period contains the purchase date, and brings in that row’s customer_key. It then checks that every order matched exactly once.

In [0]:
orders = silver("orders")
items = silver("order_items")

d_cust = gold("dim_customer").select("customer_key", "customer_id", "start_date", "end_date")
d_sell = gold("dim_seller").select("seller_key", "seller_id")
d_prod = gold("dim_product").select("product_key", "product_id")

date_key = lambda c: F.date_format(c, "yyyyMMdd").cast("int")

orders_k = (orders.alias("o")
    .join(d_cust.alias("c"),
          (F.col("o.customer_id") == F.col("c.customer_id")) &
          (F.to_date("o.order_purchase_timestamp") >= F.col("c.start_date")) &
          (F.to_date("o.order_purchase_timestamp") < F.col("c.end_date")),
          "left")
    .select("o.*", "c.customer_key"))

print("silver orders:", orders.count(),
      "| after join:", orders_k.count(),
      "| missing customer_key:", orders_k.filter(F.col("customer_key").isNull()).count())

silver orders: 66638 | after join: 66638 | missing customer_key: 0


fact_order_items

This takes each silver order item, attaches the customer key, the seller key, and the product key, adds purchase_date_key and item_total (price + freight), and merges the result on order_id + order_item_id.

In [0]:
fact_order_items = (items
    .join(orders_k.select("order_id", "customer_key", "order_status", "order_purchase_timestamp"), "order_id")
    .join(d_sell, "seller_id", "left")
    .join(d_prod, "product_id", "left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("item_total", F.round(F.col("price") + F.col("freight_value"), 2))
    .select("order_id", "order_item_id", "customer_key", "seller_key", "product_key",
            "purchase_date_key", "order_status", "price", "freight_value", "item_total"))
merge_into_gold(fact_order_items, "fact_order_items", ["order_id", "order_item_id"])

fact_order_items: inserted 0, updated 75331, table total 75331


fact_orders

This builds the three per-order summaries (items, payments, reviews), attaches them to each order along with its customer key, calculates the delivery measures, and merges on order_id.

In [0]:
item_agg = items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.round(F.sum("price"), 2).alias("items_value"),
    F.round(F.sum("freight_value"), 2).alias("freight_value"))

pay_agg = silver("order_payments").groupBy("order_id").agg(
    F.round(F.sum("payment_value"), 2).alias("payment_value"),
    F.max("payment_installments").alias("max_installments"))

rev_agg = silver("order_reviews").groupBy("order_id").agg(
    F.round(F.avg("review_score"), 2).alias("avg_review_score"))

fact_orders = (orders_k
    .join(item_agg, "order_id", "left")
    .join(pay_agg, "order_id", "left")
    .join(rev_agg, "order_id", "left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("estimated_delivery_days", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
    .withColumn("delivery_delay_days", F.datediff("order_delivered_customer_date", "order_estimated_delivery_date"))
    .withColumn("is_late", F.col("delivery_delay_days") > 0)
    .select("order_id", "customer_key", "purchase_date_key", "order_status",
            "items_count", "items_value", "freight_value", "payment_value", "max_installments",
            "avg_review_score", "delivery_days", "estimated_delivery_days",
            "delivery_delay_days", "is_late"))
merge_into_gold(fact_orders, "fact_orders", ["order_id"])

fact_orders: inserted 0, updated 66638, table total 66638


checks

For both fact tables, this prints the total, the number of duplicate keys, and the number of missing keys. It then proves the SCD history is preserved by counting orders linked to the new versions (keys 99442 to 99444) and the old versions (keys 2, 3, 4) of the three customers who moved.

In [0]:
checks = {"fact_order_items": ["order_id", "order_item_id"], "fact_orders": ["order_id"]}
for t, k in checks.items():
    tbl = gold(t)
    dups = tbl.groupBy(*k).count().filter("count > 1").count()
    nulls = tbl.filter(F.col("customer_key").isNull()).count()
    print(f"{t}: total {tbl.count()} | duplicate keys: {dups} | missing customer_key: {nulls}")

fi = gold("fact_order_items")
print("items missing seller or product key:",
      fi.filter(F.col("seller_key").isNull() | F.col("product_key").isNull()).count())

fo = gold("fact_orders")
print("orders on NEW versions of moved customers:", fo.filter(F.col("customer_key").isin(99442, 99443, 99444)).count())
print("orders on OLD versions of moved customers:", fo.filter(F.col("customer_key").isin(2, 3, 4)).count())

fact_order_items: total 75331 | duplicate keys: 0 | missing customer_key: 0
fact_orders: total 66638 | duplicate keys: 0 | missing customer_key: 0
items missing seller or product key: 0
orders on NEW versions of moved customers: 0
orders on OLD versions of moved customers: 3
